In [1]:
import os
from pathlib import Path

In [2]:
# --- Basic local project config ---
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
!pip install -q requests tqdm

In [4]:
import re
import time
import requests
import pandas as pd
from io import StringIO
from pathlib import Path
from tqdm.notebook import tqdm
from requests.adapters import HTTPAdapter, Retry

# Data Fetching

In [5]:
class UniProtTSVFetcher:
    """
    Fetch UniProtKB data in TSV format with pagination.
    """

    BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

    def __init__(self, max_proteins=10000, batch_size=500, sleep_time=0.3):
        self.max_proteins = max_proteins
        self.batch_size = batch_size
        self.sleep_time = sleep_time
        self.df = pd.DataFrame()

        self.session = requests.Session()

        retries = Retry(
            total=5,
            backoff_factor=0.5,
            status_forcelist=[500, 502, 503, 504],
            allowed_methods=["GET"]
        )

        self.session.mount("https://", HTTPAdapter(max_retries=retries))

    def _get_next_link(self, response):
        """
        Extract the next page URL from the Link header.

        UniProt returns something like:
        Link: <https://rest.uniprot.org/uniprotkb/search?...&cursor=...>; rel="next"
        """

        # First try requests' built-in link parser
        if response.links and "next" in response.links:
            return response.links["next"]["url"]

        # Fallback regex parser
        link_header = response.headers.get("Link", "")
        match = re.search(r'<([^>]+)>;\s*rel="next"', link_header)

        if match:
            return match.group(1)

        return None

    def _read_tsv_response(self, response):
        """
        Convert TSV text response into a pandas DataFrame.
        """
        if not response.text.strip():
            return pd.DataFrame()

        return pd.read_csv(StringIO(response.text), sep="\t")

    def fetch_proteins(self, query="reviewed:true AND organism_id:9606"):
        """
        Fetch proteins from UniProtKB in TSV format.
        """

        print("Fetching proteins from UniProt...")
        print(f"Query: {query}")
        print(f"Target proteins: {self.max_proteins}")
        print(f"Batch size: {self.batch_size}\n")

        params = {
            "query": query,
            "format": "tsv",
            "size": self.batch_size,
            "fields": ",".join([
                "accession",
                "id",
                "protein_name",
                "gene_names",
                "organism_name",
                "length",
                "sequence",
                "cc_function",
                "go_id",
                "go",
                "ec",
                "keyword"
            ])
        }

        all_batches = []
        next_url = self.BASE_URL
        total_downloaded = 0
        first_request = True

        with tqdm(total=self.max_proteins, desc="Downloading", unit="proteins") as pbar:

            while next_url and total_downloaded < self.max_proteins:

                if first_request:
                    response = self.session.get(
                        next_url,
                        params=params,
                        timeout=60
                    )
                    first_request = False
                else:
                    # next_url already includes query params and cursor
                    response = self.session.get(
                        next_url,
                        timeout=60
                    )

                response.raise_for_status()

                batch_df = self._read_tsv_response(response)

                if batch_df.empty:
                    print("No more rows returned.")
                    break

                remaining = self.max_proteins - total_downloaded

                if len(batch_df) > remaining:
                    batch_df = batch_df.head(remaining)

                all_batches.append(batch_df)

                total_downloaded += len(batch_df)
                pbar.update(len(batch_df))

                next_url = self._get_next_link(response)

                if not next_url:
                    print("\nReached last page. No next URL found.")
                    break

                time.sleep(self.sleep_time)

        if all_batches:
            self.df = pd.concat(all_batches, ignore_index=True)

            # Drop duplicate proteins based on UniProt accession if present
            if "Entry" in self.df.columns:
                before = len(self.df)
                self.df = self.df.drop_duplicates(subset=["Entry"]).reset_index(drop=True)
                after = len(self.df)

                print(f"\nRemoved duplicates: {before - after}")

        print(f"\nFetched {len(self.df)} unique proteins.")

        return self.df

    def save_tsv(self, filepath):
        filepath = Path(filepath)
        filepath.parent.mkdir(parents=True, exist_ok=True)

        self.df.to_csv(filepath, sep="\t", index=False)
        print(f"Saved TSV to: {filepath}")

    def save_csv(self, filepath):
        filepath = Path(filepath)
        filepath.parent.mkdir(parents=True, exist_ok=True)

        self.df.to_csv(filepath, index=False)
        print(f"Saved CSV to: {filepath}")

    def verify_uniqueness(self):
        if self.df.empty:
            print("No data available. Run fetch_proteins() first.")
            return False

        if "Entry" not in self.df.columns:
            print("Column 'Entry' not found. Cannot verify accession uniqueness.")
            return False

        total = len(self.df)
        unique = self.df["Entry"].nunique()
        duplicates = total - unique

        print("\nUniqueness verification")
        print(f"Total rows: {total}")
        print(f"Unique UniProt accessions: {unique}")
        print(f"Duplicate rows: {duplicates}")

        return duplicates == 0

In [6]:
MAX_PROTEINS = 10000

fetcher = UniProtTSVFetcher(
    max_proteins=MAX_PROTEINS,
    batch_size=500,
    sleep_time=0.3
)

df = fetcher.fetch_proteins(
    query="reviewed:true AND organism_id:9606"
)

df.head()

Fetching proteins from UniProt...
Query: reviewed:true AND organism_id:9606
Target proteins: 10000
Batch size: 500



Downloading:   0%|          | 0/10000 [00:00<?, ?proteins/s]


Removed duplicates: 0

Fetched 10000 unique proteins.


,Entry,Entry Name,Protein names,Gene Names,Organism,Length,Sequence,Function [CC],Gene Ontology IDs,Gene Ontology (GO),EC number,Keywords
0,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c (Mitochon...,MT-RNR1,Homo sapiens (Human),16,MRWQEMGYIFYPRKLR,FUNCTION: Regulates insulin sensitivity and me...,GO:0001649; GO:0003677; GO:0005615; GO:0005634...,extracellular space [GO:0005615]; mitochondrio...,NaN,DNA-binding;Mitochondrion;Nucleus;Osteogenesis...
1,A0A1B0GTW7,CIROP_HUMAN,Ciliated left-right organizer metallopeptidase...,CIROP LMLN2,Homo sapiens (Human),788,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,FUNCTION: Putative metalloproteinase that play...,GO:0004222; GO:0005737; GO:0006508; GO:0007155...,cytoplasm [GO:0005737]; membrane [GO:0016020];...,3.4.24.-,Alternative splicing;Disease variant;Glycoprot...
2,A0JNW5,BLT3B_HUMAN,Bridge-like lipid transfer protein family memb...,BLTP3B KIAA0701 SHIP164 UHRF1BP1L,Homo sapiens (Human),1464,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,FUNCTION: Tube-forming lipid transport protein...,GO:0005769; GO:0005829; GO:0034498; GO:0042803...,cytosol [GO:0005829]; early endosome [GO:00057...,NaN,Alternative splicing;Coiled coil;Cytoplasm;End...
3,A0JP26,POTB3_HUMAN,POTE ankyrin domain family member B3,POTEB3,Homo sapiens (Human),581,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,NaN,NaN,NaN,NaN,Alternative splicing;ANK repeat;Coiled coil;Re...
4,A0PK11,CLRN2_HUMAN,Clarin-2,CLRN2,Homo sapiens (Human),232,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,FUNCTION: Plays a key role to hearing function...,GO:0007605; GO:0032421; GO:0060088; GO:0060171...,stereocilium bundle [GO:0032421]; stereocilium...,NaN,Cell membrane;Cell projection;Deafness;Disease...


In [7]:
#Verify Accession ID Uniqueness

is_unique = fetcher.verify_uniqueness()


Uniqueness verification
Total rows: 10000
Unique UniProt accessions: 10000
Duplicate rows: 0


### Save raw files

In [8]:
raw_tsv_file = RAW_DIR / "uniprot_proteins.tsv"
raw_csv_file = RAW_DIR / "uniprot_proteins.csv"

fetcher.save_tsv(raw_tsv_file)
fetcher.save_csv(raw_csv_file)

Saved TSV to: C:\Users\sridi\Documents\Learning\protein-function-active-learning\data\raw\uniprot_proteins.tsv
Saved CSV to: C:\Users\sridi\Documents\Learning\protein-function-active-learning\data\raw\uniprot_proteins.csv


### Dataset statistics

In [9]:
print("\nDataset Statistics")
print(f"Total proteins: {len(df)}")

if "Length" in df.columns:
    print(f"Average sequence length: {df['Length'].mean():.0f}")
    print(f"Median sequence length: {df['Length'].median():.0f}")

if "Function [CC]" in df.columns:
    proteins_with_function = df["Function [CC]"].notna().sum()
    print(f"Proteins with function annotation: {proteins_with_function}")

if "Gene Ontology IDs" in df.columns:
    proteins_with_go = df["Gene Ontology IDs"].notna().sum()
    print(f"Proteins with GO terms: {proteins_with_go}")

if "EC number" in df.columns:
    proteins_with_ec = df["EC number"].notna().sum()
    print(f"Proteins with EC numbers: {proteins_with_ec}")

print("\nColumns returned:")
print(df.columns.tolist())


Dataset Statistics
Total proteins: 10000
Average sequence length: 617
Median sequence length: 459
Proteins with function annotation: 9676
Proteins with GO terms: 9968
Proteins with EC numbers: 2899

Columns returned:
['Entry', 'Entry Name', 'Protein names', 'Gene Names', 'Organism', 'Length', 'Sequence', 'Function [CC]', 'Gene Ontology IDs', 'Gene Ontology (GO)', 'EC number', 'Keywords']


### Verify a sample protein

In [10]:
# Show sample protein
sample = df.iloc[0]

print("\nSample Protein")

for col in df.columns:
    value = sample[col]

    if isinstance(value, str) and len(value) > 300:
        value = value[:300] + "..."

    print(f"{col}: {value}")



Sample Protein
Entry: A0A0C5B5G6
Entry Name: MOTSC_HUMAN
Protein names: Mitochondrial-derived peptide MOTS-c (Mitochondrial open reading frame of the 12S rRNA-c)
Gene Names: MT-RNR1
Organism: Homo sapiens (Human)
Length: 16
Sequence: MRWQEMGYIFYPRKLR
Function [CC]: FUNCTION: Regulates insulin sensitivity and metabolic homeostasis (PubMed:25738459, PubMed:33468709). Inhibits the folate cycle, thereby reducing de novo purine biosynthesis which leads to the accumulation of the de novo purine synthesis intermediate 5-aminoimidazole-4-carboxamide (AICAR) and the ac...
Gene Ontology IDs: GO:0001649; GO:0003677; GO:0005615; GO:0005634; GO:0005739; GO:0006357; GO:0032147; GO:0033687; GO:0043610; GO:0048630; GO:0071902; GO:0072522; GO:0140297; GO:2001145
Gene Ontology (GO): extracellular space [GO:0005615]; mitochondrion [GO:0005739]; nucleus [GO:0005634]; DNA binding [GO:0003677]; DNA-binding transcription factor binding [GO:0140297]; activation of protein kinase activity [GO:0032147]; negati

### Standardize column names

In [11]:
def standardize_uniprot_columns(df):
    """
    Rename UniProt TSV columns into simpler project-friendly names.
    """

    rename_map = {
        "Entry": "accession",
        "Entry Name": "entry_name",
        "Protein names": "protein_name",
        "Gene Names": "gene_names",
        "Organism": "organism",
        "Length": "seq_length",
        "Sequence": "sequence",
        "Function [CC]": "function",
        "Gene Ontology IDs": "go_ids",
        "Gene Ontology (GO)": "go_terms",
        "EC number": "ec_numbers",
        "Keywords": "keywords"
    }

    existing_rename_map = {
        old: new for old, new in rename_map.items()
        if old in df.columns
    }

    df_clean = df.rename(columns=existing_rename_map).copy()

    return df_clean

In [12]:
df_clean = standardize_uniprot_columns(df)

processed_file = PROCESSED_DIR / "uniprot_proteins_clean.csv"
df_clean.to_csv(processed_file, index=False)

print(f"Saved cleaned file to: {processed_file}")
df_clean.head()

Saved cleaned file to: C:\Users\sridi\Documents\Learning\protein-function-active-learning\data\processed\uniprot_proteins_clean.csv


,accession,entry_name,protein_name,gene_names,organism,seq_length,sequence,function,go_ids,go_terms,ec_numbers,keywords
0,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c (Mitochon...,MT-RNR1,Homo sapiens (Human),16,MRWQEMGYIFYPRKLR,FUNCTION: Regulates insulin sensitivity and me...,GO:0001649; GO:0003677; GO:0005615; GO:0005634...,extracellular space [GO:0005615]; mitochondrio...,NaN,DNA-binding;Mitochondrion;Nucleus;Osteogenesis...
1,A0A1B0GTW7,CIROP_HUMAN,Ciliated left-right organizer metallopeptidase...,CIROP LMLN2,Homo sapiens (Human),788,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,FUNCTION: Putative metalloproteinase that play...,GO:0004222; GO:0005737; GO:0006508; GO:0007155...,cytoplasm [GO:0005737]; membrane [GO:0016020];...,3.4.24.-,Alternative splicing;Disease variant;Glycoprot...
2,A0JNW5,BLT3B_HUMAN,Bridge-like lipid transfer protein family memb...,BLTP3B KIAA0701 SHIP164 UHRF1BP1L,Homo sapiens (Human),1464,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,FUNCTION: Tube-forming lipid transport protein...,GO:0005769; GO:0005829; GO:0034498; GO:0042803...,cytosol [GO:0005829]; early endosome [GO:00057...,NaN,Alternative splicing;Coiled coil;Cytoplasm;End...
3,A0JP26,POTB3_HUMAN,POTE ankyrin domain family member B3,POTEB3,Homo sapiens (Human),581,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,NaN,NaN,NaN,NaN,Alternative splicing;ANK repeat;Coiled coil;Re...
4,A0PK11,CLRN2_HUMAN,Clarin-2,CLRN2,Homo sapiens (Human),232,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,FUNCTION: Plays a key role to hearing function...,GO:0007605; GO:0032421; GO:0060088; GO:0060171...,stereocilium bundle [GO:0032421]; stereocilium...,NaN,Cell membrane;Cell projection;Deafness;Disease...


In [13]:
print(f"Requested proteins: {MAX_PROTEINS}")
print(f"Downloaded proteins: {len(df_clean)}")

Requested proteins: 10000
Downloaded proteins: 10000


# Data Filtering

In [14]:
df_filtered = df_clean.copy()

print("Before filtering:", df_filtered.shape)

Before filtering: (10000, 12)


In [15]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   accession     10000 non-null  object
 1   entry_name    10000 non-null  object
 2   protein_name  10000 non-null  object
 3   gene_names    9997 non-null   object
 4   organism      10000 non-null  object
 5   seq_length    10000 non-null  int64 
 6   sequence      10000 non-null  object
 7   function      9676 non-null   object
 8   go_ids        9968 non-null   object
 9   go_terms      9968 non-null   object
 10  ec_numbers    2899 non-null   object
 11  keywords      10000 non-null  object
dtypes: int64(1), object(11)
memory usage: 937.6+ KB


In [16]:
# 1. Required fields
df_filtered = df_filtered.dropna(subset=["accession", "sequence"])

# 2. Remove duplicate UniProt accessions - already done during fetching
# df_filtered = df_filtered.drop_duplicates(subset=["accession"])

# 3. Make sequence uppercase
df_filtered["sequence"] = df_filtered["sequence"].astype(str).str.upper()

# 4. Remove invalid or ambiguous amino acid sequences
# Standard 20 amino acids only
valid_aa_pattern = r"^[ACDEFGHIKLMNPQRSTVWY]+$"
df_filtered = df_filtered[
    df_filtered["sequence"].str.match(valid_aa_pattern)
]

# 5. Length filter : Keep proteins between 50 and 1000 amino acids long
df_filtered = df_filtered[
    (df_filtered["seq_length"] >= 50) &
    (df_filtered["seq_length"] <= 1000)
]

# 6. Remove fragments if protein name contains "Fragment"
df_filtered = df_filtered[
    ~df_filtered["protein_name"].fillna("").str.contains("fragment", case=False, na=False)
]

# 7. Keep proteins with at least one useful annotation
has_annotation = (
    df_filtered["go_terms"].fillna("").str.len().gt(0) |
    df_filtered["keywords"].fillna("").str.len().gt(0) |
    df_filtered["ec_numbers"].fillna("").str.len().gt(0)
)

df_filtered = df_filtered[has_annotation]

In [17]:
print("After filtering:", df_filtered.shape)
df_filtered.to_csv(PROCESSED_DIR / "uniprot_proteins_filtered.csv", index=False)

After filtering: (8520, 12)


# Assign broad function classes (target label)

Build the single-label target column `function_class` used for all supervised
modeling and active learning. Each protein is assigned exactly one of:

    enzyme, transporter, receptor, dna_rna_binding, structural, other

Because a protein can match several functional descriptions at once (e.g. a
membrane-bound kinase), we resolve overlaps with an explicit **priority
cascade**. The order below is a deliberate choice and should be reported:

    1. enzyme           (strongest signal: EC number / catalytic terms)
    2. dna_rna_binding
    3. receptor
    4. transporter
    5. structural
    6. other            (no rule matched)

Note: This label is derived from UniProt keywords / GO terms / EC numbers, so the
classifier partly learns to reproduce UniProt's own annotation rules from
sequence. 

In [18]:
df_labeled = df_filtered.copy()

for col in ["protein_name", "function", "go_terms", "keywords", "ec_numbers"]:
    if col in df_labeled.columns:
        df_labeled[col] = df_labeled[col].fillna("").astype(str)
    else:
        df_labeled[col] = ""


def is_enzyme(row):
    return (
        len(row["ec_numbers"]) > 0
        or bool(re.search(r"enzyme|hydrolase|transferase|oxidoreductase|kinase|protease|ligase|lyase|isomerase",
                          row["keywords"], re.IGNORECASE))
        or bool(re.search(r"catalytic activity|hydrolase activity|transferase activity|kinase activity",
                          row["go_terms"], re.IGNORECASE))
    )

def is_dna_rna_binding(row):
    return (
        bool(re.search(r"DNA-binding|RNA-binding|nucleotide-binding", row["keywords"], re.IGNORECASE))
        or bool(re.search(r"DNA binding|RNA binding|nucleic acid binding|transcription factor",
                          row["go_terms"], re.IGNORECASE))
        or bool(re.search(r"DNA-binding|RNA-binding|transcription factor|zinc finger",
                          row["protein_name"], re.IGNORECASE))
    )

def is_receptor(row):
    return (
        bool(re.search(r"receptor", row["keywords"], re.IGNORECASE))
        or bool(re.search(r"receptor activity|signaling receptor activity", row["go_terms"], re.IGNORECASE))
        or bool(re.search(r"receptor", row["protein_name"], re.IGNORECASE))
    )

def is_transporter(row):
    return (
        bool(re.search(r"transmembrane|transporter|transport", row["keywords"], re.IGNORECASE))
        or bool(re.search(r"transporter activity|transmembrane transport", row["go_terms"], re.IGNORECASE))
        or bool(re.search(r"transporter|solute carrier|channel|pump", row["protein_name"], re.IGNORECASE))
    )

def is_structural(row):
    return (
        bool(re.search(r"cytoskeleton|structural|collagen|keratin|extracellular matrix",
                       row["keywords"], re.IGNORECASE))
        or bool(re.search(r"structural molecule activity|cytoskeleton|extracellular matrix",
                          row["go_terms"], re.IGNORECASE))
        or bool(re.search(r"collagen|keratin|tubulin|actin|myosin|laminin",
                          row["protein_name"], re.IGNORECASE))
    )

In [19]:
def assign_function_class(row):
    """Single-label priority cascade. First match wins."""
    if is_enzyme(row):
        return "enzyme"
    if is_dna_rna_binding(row):
        return "dna_rna_binding"
    if is_receptor(row):
        return "receptor"
    if is_transporter(row):
        return "transporter"
    if is_structural(row):
        return "structural"
    return "other"


df_labeled["function_class"] = df_labeled.apply(assign_function_class, axis=1)

print("Class distribution:")
counts = df_labeled["function_class"].value_counts()
for cls, n in counts.items():
    print(f"  {cls:16s} {n:5d}  ({n / len(df_labeled):.1%})")
print(f"\nTotal labeled proteins: {len(df_labeled)}")

Class distribution:
  enzyme            2687  (31.5%)
  dna_rna_binding   1723  (20.2%)
  other             1304  (15.3%)
  transporter       1300  (15.3%)
  receptor           758  (8.9%)
  structural         748  (8.8%)

Total labeled proteins: 8520


In [27]:
labeled_file = PROCESSED_DIR / "labeled_dataset.csv"
df_labeled.to_csv(labeled_file, index=False)
print(f"Saved labeled dataset to: {labeled_file}")
print(f"Columns: {df_labeled.columns.tolist()}")

Saved labeled dataset to: C:\Users\sridi\Documents\Learning\protein-function-active-learning\data\processed\labeled_dataset.csv
Columns: ['accession', 'entry_name', 'protein_name', 'gene_names', 'organism', 'seq_length', 'sequence', 'function', 'go_ids', 'go_terms', 'ec_numbers', 'keywords', 'function_class']


# FASTA and protein ID export for Galaxy

In [20]:
GALAXY_INPUT_DIR = PROJECT_ROOT / "galaxy_inputs"
GALAXY_INPUT_DIR.mkdir(parents=True, exist_ok=True)

protein_id_file = GALAXY_INPUT_DIR / "protein_ids.txt"
fasta_file = GALAXY_INPUT_DIR / "protein_sequences.fasta"

# Protein IDs for Galaxy ID mapping / GO / Reactome workflows
with open(protein_id_file, "w", encoding="utf-8") as f:
    for acc in df_filtered["accession"].dropna().unique():
        f.write(f"{acc}\n")

# FASTA file for Galaxy sequence-based annotation tools
with open(fasta_file, "w", encoding="utf-8") as f:
    for _, row in df_filtered.iterrows():
        accession = row.get("accession", "")
        protein_name = str(row.get("protein_name", "")).replace("\n", " ")
        sequence = row.get("sequence", "")

        if not accession or not isinstance(sequence, str) or not sequence:
            continue

        f.write(f">{accession} {protein_name}\n")

        for i in range(0, len(sequence), 80):
            f.write(sequence[i:i+80] + "\n")

print(f"Saved Galaxy protein IDs to: {protein_id_file}")
print(f"Saved Galaxy FASTA file to: {fasta_file}")

Saved Galaxy protein IDs to: C:\Users\sridi\Documents\Learning\protein-function-active-learning\galaxy_inputs\protein_ids.txt
Saved Galaxy FASTA file to: C:\Users\sridi\Documents\Learning\protein-function-active-learning\galaxy_inputs\protein_sequences.fasta


# Create subsets from cleaned CSV - defining annotation-based subsets

In [21]:
SUBSET_DIR = PROJECT_ROOT / "galaxy_inputs" / "subsets"
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

df = df_filtered.copy()

# Make text columns safe
for col in ["protein_name", "function", "go_terms", "keywords", "ec_numbers"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)
    else:
        df[col] = ""

def save_subset(df_subset, filename, max_n=300):
    """
    Save UniProt accessions to a text file, one accession per line.
    """
    accessions = (
        df_subset["accession"]
        .dropna()
        .drop_duplicates()
        .head(max_n)
    )

    output_path = SUBSET_DIR / filename

    with open(output_path, "w", encoding="utf-8") as f:
        for acc in accessions:
            f.write(f"{acc}\n")

    print(f"{filename}: {len(accessions)} proteins saved")
    return output_path

## Enzymes
Use EC numbers and enzyme-related keywords.

In [22]:
enzyme_mask = (
    df["ec_numbers"].str.len().gt(0)
    | df["keywords"].str.contains("enzyme|hydrolase|transferase|oxidoreductase|kinase|protease", case=False, na=False)
    | df["go_terms"].str.contains("catalytic activity|hydrolase activity|transferase activity|kinase activity", case=False, na=False)
)

save_subset(df[enzyme_mask], "enzyme_ids.txt", max_n=300)

enzyme_ids.txt: 300 proteins saved


WindowsPath('C:/Users/sridi/Documents/Learning/protein-function-active-learning/galaxy_inputs/subsets/enzyme_ids.txt')

## Transporters

In [23]:
transporter_mask = (
    df["keywords"].str.contains("transmembrane|transporter|transport", case=False, na=False)
    | df["go_terms"].str.contains("transporter activity|transport|transmembrane transport", case=False, na=False)
    | df["protein_name"].str.contains("transporter|solute carrier|channel|pump", case=False, na=False)
)

save_subset(df[transporter_mask], "transporter_ids.txt", max_n=300)

transporter_ids.txt: 300 proteins saved


WindowsPath('C:/Users/sridi/Documents/Learning/protein-function-active-learning/galaxy_inputs/subsets/transporter_ids.txt')

## Receptors

In [24]:
receptor_mask = (
    df["keywords"].str.contains("receptor", case=False, na=False)
    | df["go_terms"].str.contains("receptor activity|signaling receptor activity", case=False, na=False)
    | df["protein_name"].str.contains("receptor", case=False, na=False)
)

save_subset(df[receptor_mask], "receptor_ids.txt", max_n=300)

receptor_ids.txt: 300 proteins saved


WindowsPath('C:/Users/sridi/Documents/Learning/protein-function-active-learning/galaxy_inputs/subsets/receptor_ids.txt')

## DNA/RNA-binding proteins

In [25]:
dna_rna_binding_mask = (
    df["keywords"].str.contains("DNA-binding|RNA-binding|nucleotide-binding", case=False, na=False)
    | df["go_terms"].str.contains("DNA binding|RNA binding|nucleic acid binding|transcription factor", case=False, na=False)
    | df["protein_name"].str.contains("DNA-binding|RNA-binding|transcription factor|zinc finger", case=False, na=False)
)

save_subset(df[dna_rna_binding_mask], "dna_rna_binding_ids.txt", max_n=300)

dna_rna_binding_ids.txt: 300 proteins saved


WindowsPath('C:/Users/sridi/Documents/Learning/protein-function-active-learning/galaxy_inputs/subsets/dna_rna_binding_ids.txt')

## Create a background file

In [26]:
save_subset(df, "background_all_ids.txt", max_n=len(df))

background_all_ids.txt: 8520 proteins saved


WindowsPath('C:/Users/sridi/Documents/Learning/protein-function-active-learning/galaxy_inputs/subsets/background_all_ids.txt')